# Speed Dating Model
I wanted to start playing around with the data and I thought a Notebook file would be a good place to do so. I'm happy to migrate it to an actual Python script at some point, this can just be a testing ground if necessary.

In [5]:
# Imports
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score

# Load the data
df = pd.read_csv('./data/Speed Dating Data.csv', encoding='cp1252')

In [6]:
columns_to_keep = np.r_[
    0, # iid
    11, # pid
    12, # match (VERY IMPORTANT THIS IS WHAT WE'RE TRYING TO PREDICT)
    33, # age
    45, # goal, NOTE would need one-hot encoding
    49, # career_c, NOTE would need one-hot encoding
    50:67, # interests, rated
]

filtered_df = df.iloc[:, columns_to_keep]
filtered_df.head()

,iid,pid,match,age,goal,career_c,sports,tvsports,exercise,dining,...,gaming,clubbing,reading,tv,theater,movies,concerts,music,shopping,yoga
0,1,11.0,0,21.0,2.0,NaN,9.0,2.0,8.0,9.0,...,1.0,5.0,6.0,9.0,1.0,10.0,10.0,9.0,8.0,1.0
1,1,12.0,0,21.0,2.0,NaN,9.0,2.0,8.0,9.0,...,1.0,5.0,6.0,9.0,1.0,10.0,10.0,9.0,8.0,1.0
2,1,13.0,1,21.0,2.0,NaN,9.0,2.0,8.0,9.0,...,1.0,5.0,6.0,9.0,1.0,10.0,10.0,9.0,8.0,1.0
3,1,14.0,1,21.0,2.0,NaN,9.0,2.0,8.0,9.0,...,1.0,5.0,6.0,9.0,1.0,10.0,10.0,9.0,8.0,1.0
4,1,15.0,1,21.0,2.0,NaN,9.0,2.0,8.0,9.0,...,1.0,5.0,6.0,9.0,1.0,10.0,10.0,9.0,8.0,1.0


In [7]:
# One-hot encode categorical data
one_hot_goal = pd.get_dummies(filtered_df['goal'])
one_hot_goal.rename(inplace=True, axis="columns", mapper={
    1: 'joined_fun',
    2: 'joined_meet',
    3: 'joined_date',
    4: 'joined_relationship',
    5: 'joined_to_try',
    6: 'joined_other'
})

one_hot_career = pd.get_dummies(filtered_df['career_c'])
one_hot_career.rename(inplace=True, axis="columns", mapper={
    1: 'career_lawyer',
    2: 'career_academia',
    3: 'career_psychologist',
    4: 'career_medicine',
    5: 'career_engineer',
    6: 'career_creative_arts',
    7: 'career_business',
    8: 'career_real_estate',
    9: 'career_humanitarian_affairs',
    10: 'career_undecided',
    11: 'career_social_work',
    12: 'career_speech_pathology',
    13: 'career_politics',
    14: 'career_pro_sports',
    15: 'career_other',
    16: 'career_journalism',
    17: 'career_architecture'
})

encoded_df = filtered_df.drop(['goal', 'career_c'], axis=1)
encoded_df = pd.concat([encoded_df, one_hot_goal, one_hot_career], axis=1)

# 
cleaned_df = encoded_df.merge(
    encoded_df,
    left_on=['iid', 'pid'],
    right_on=['pid', 'iid'],
    suffixes=('', '_partner')
).drop(['pid_partner', 'iid_partner', 'match_partner'], axis=1)
cleaned_df = cleaned_df[cleaned_df['iid'] < cleaned_df['pid']]
print(cleaned_df.columns)

Index(['iid', 'pid', 'match', 'age', 'sports', 'tvsports', 'exercise',
       'dining', 'museums', 'art', 'hiking', 'gaming', 'clubbing', 'reading',
       'tv', 'theater', 'movies', 'concerts', 'music', 'shopping', 'yoga',
       'joined_fun', 'joined_meet', 'joined_date', 'joined_relationship',
       'joined_to_try', 'joined_other', 'career_lawyer', 'career_academia',
       'career_psychologist', 'career_medicine', 'career_engineer',
       'career_creative_arts', 'career_business', 'career_real_estate',
       'career_humanitarian_affairs', 'career_undecided', 'career_social_work',
       'career_speech_pathology', 'career_politics', 'career_pro_sports',
       'career_other', 'career_journalism', 'career_architecture',
       'age_partner', 'sports_partner', 'tvsports_partner', 'exercise_partner',
       'dining_partner', 'museums_partner', 'art_partner', 'hiking_partner',
       'gaming_partner', 'clubbing_partner', 'reading_partner', 'tv_partner',
       'theater_partner', 'mov

In [8]:
# model goes here
# starting with random forest but I have very little reason behind that choice so suggestions welcome
X = cleaned_df.drop(['match', 'iid', 'pid'], axis=1)
y = cleaned_df['match']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4)

classifier = RandomForestClassifier()
classifier.fit(X_train, y_train)

# Test model
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

y_pred_proba = classifier.predict_proba(X_test)
loss = log_loss(y_test, y_pred_proba)
print (f"Loss: {loss}")

Accuracy: 0.8422939068100358
Loss: 0.4174123162523687


In [9]:
# Save model
import pickle
with open('model.pkl', 'wb') as f:
    pickle.dump(classifier,f)